# Phase 6b: is the steering effect specific to v? (Qwen2.5-7B-Instruct, bf16)

**A100**, then `Runtime → Run all`. Needs Phase 5 `vectors.pt` and Phase 6 `steer/` on Drive.

Phase 6 found that v moves flips ~3 points, but so did one random vector (opposite sign) and the shuffled vector. Two tests:

1. **Null distribution** (36 tuning questions): 19 more random directions of the same length as v at layers 16 and 18, λ = +1,
   plus shuffled. Is v's effect bigger than a typical random push? (empirical p over 20 random directions)
2. **Directional ablation** (143 final questions): remove the v component entirely, h ← h − (h·û)û, at layer 18 or at every layer,
   on every token. If v carries the rank information the model uses, the senior − junior gap should shrink;
   ablating random or shuffled directions should not.

Resumable: every unit is saved to Drive as it finishes; if Colab drops, `Run all` again.

In [ ]:
CFG = "configs/colab.yaml"
AUTO_DISCONNECT = True
BS = 64
N_RANDOM = 19          # random1..random19, plus Phase 6's `random` = 20 directions
NULL_LAYERS = "16 18"
ABL_RANDOM = 4         # random, random1..4 = 5 random directions for ablation

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f"https://{token}@github.com/GTRAN7/vnpronouns.git" if token else "https://github.com/GTRAN7/vnpronouns.git"
if not os.path.exists('/content/vnpronouns'):
    subprocess.run(['git', 'clone', '-q', url, '/content/vnpronouns'], check=True)
else:
    subprocess.run(['git', '-C', '/content/vnpronouns', 'pull', '-q'], check=True)
%cd /content/vnpronouns
!git log --oneline -1
!pip install -q -e .

def run(cmd, log, env=None):
    """Run a shell command, stream + log its output, and raise if it fails (so Run all stops).
    Splits only on real newlines and keeps the text after the last carriage return, so tqdm
    progress bars collapse to one line instead of flooding the notebook."""
    print(f'$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1', **(env or {})})
    with open(log, 'a', encoding='utf-8') as f:
        for raw in p.stdout:
            line = raw.decode('utf-8', 'replace').rstrip('\n').split('\r')[-1]
            if line.strip():
                print(line, flush=True)
                f.write(line + '\n')
    if p.wait() != 0:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}  -- see {log}')

## 1. GPU check + smoke test

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> A100 (or L4)'
name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'{name}, {mem:.0f} GB, bf16 supported: {torch.cuda.is_bf16_supported()}')
assert torch.cuda.is_bf16_supported() and mem >= 22, f'{name} cannot hold the 7B in bf16; switch to A100/L4'

import os, shutil, time
OUT = '/content/drive/MyDrive/vnpronouns/outputs/Qwen2.5-7B-Instruct'
assert os.path.exists(f'{OUT}/vectors.pt'), 'vectors.pt missing: run colab_7b_direction.ipynb first'
assert os.path.exists(f'{OUT}/steer/tune/none_L-1_lam+0.csv'), 'Phase 6 tune units missing: run colab_7b_steering.ipynb first'
SMOKE = {'VNP_OUTPUT_DIR': '/content/smoke'}
shutil.rmtree('/content/smoke', ignore_errors=True)
t0 = time.time()
L = '/content/log_smoke.txt'
run(f'python -m vnpronouns.run_behavior --config {CFG} --pool 2 --no-filter', L, SMOKE)
run(f'python -m vnpronouns.extract --config {CFG} --acts-path /content/smoke/acts.pt', L, SMOKE)
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split test --layers 8 --lams 1 --vectors deference random shuffled --n-random 2', L, SMOKE)
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split test --mode ablate --positions all --layers 8 all --vectors deference shuffled random --n-random 1', L, SMOKE)
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split test --mode ablate --layers all --vectors deference random', L, SMOKE)
run(f'python -m vnpronouns.steer_report --config {CFG} --split test --section null', L, SMOKE)
run(f'python -m vnpronouns.steer_report --config {CFG} --split test --section ablate --n-boot 200', L, SMOKE)
print(f'\nSMOKE TEST PASSED in {time.time() - t0:.0f}s')
shutil.rmtree('/content/smoke', ignore_errors=True)

## 2. Null distribution over random directions (tuning set)

In [ ]:
L = '/content/log_null.txt'
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split tune --layers {NULL_LAYERS} --lams 1 --vectors shuffled --n-random {N_RANDOM}', L)
!rm -f /content/log_null_report.txt
run(f'python -m vnpronouns.steer_report --config {CFG} --split tune --section null', '/content/log_null_report.txt')

## 3. Directional ablation (final set)

In [ ]:
L = '/content/log_ablate.txt'
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split final --mode ablate --positions all --layers 18 all --vectors deference shuffled random --n-random {ABL_RANDOM}', L)
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split final --mode ablate --layers all --vectors deference random --n-random 1', L)
!rm -f /content/log_ablate_report.txt
run(f'python -m vnpronouns.steer_report --config {CFG} --split final --section ablate', '/content/log_ablate_report.txt')

## 4. Summary to paste back to Claude

In [ ]:
import re, torch
def clean(path):
    lines = open(path, encoding='utf-8').read().splitlines()
    return '\n'.join(l.rstrip() for l in lines
                     if not re.search(r'it/s\]|s/it\]|Warning|warn\(|^\s*$|Loading|Fetching|Downloading|%\||^  chunk |^\$ ', l))
print(f'GPU: {torch.cuda.get_device_name(0)}  config: {CFG}')
for p in ['/content/log_null_report.txt', '/content/log_ablate_report.txt']:
    print(f'===== {p.split("/")[-1]} =====')
    print(clean(p))

In [ ]:
if AUTO_DISCONNECT:
    from google.colab import runtime
    print('Done. Disconnecting the runtime to save compute units (outputs above stay visible).')
    runtime.unassign()